# 01 — Australia economic and social exploration

**Purpose:** move from the data-quality audit to a focused question about whether Australia's material progress has coincided with stronger social well-being.

This notebook uses same-year comparisons and independent pooled survey windows. It does not impute missing years or build a weighted composite score.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.oecd_audit import (
    build_australia_summary, build_domain_coverage,
    build_same_year_comparisons, load_clean,
)

sns.set_theme(style='whitegrid', context='notebook')
DATA_FILE = PROJECT_ROOT / 'data' / 'raw' / 'OECD Data.csv'

In [ ]:
df = load_clean(DATA_FILE)
comparisons = build_same_year_comparisons(df)
australia_summary = build_australia_summary(df, comparisons, through_year=2024)
print(f'{len(df):,} rows; {df.country_code.nunique()} countries; {df.indicator_code.nunique()} indicators')
display(australia_summary)

## Domain coverage

These are coverage summaries, not rankings of domain performance. `observation_count` counts displayed rows; `independent_observation_count` prevents repeated pooled years from being mistaken for new evidence.

In [ ]:
domain_all = build_domain_coverage(df, through_year=2024)
domain_au = build_domain_coverage(df, country_code='AUS', through_year=2024)
display(domain_all)
display(domain_au)

## Economic and social indicators

The table separates direction of change from current international position. A favourable percentile near 100 means Australia compares well among countries reporting the same indicator in the same year.

In [ ]:
FOCUS_CODES = ['1_1', '1_2', '2_1', '2_2', '2_7', '3_2', '5_1', '5_3', '6_2', '7_1_DEP', '8_1_DEP', '11_1', '11_2']
focus = australia_summary.loc[australia_summary['indicator_code'].isin(FOCUS_CODES)].copy()
display(focus[[
    'indicator_code', 'indicator', 'first_year', 'last_year_through_2024',
    'first_value', 'last_value', 'percent_change', 'change_is_favourable',
    'independent_period_count', 'latest_same_year_country_count',
    'latest_favourable_percentile', 'latest_comparison_is_thin',
]])

## Direction of travel

Each indicator is indexed to 100 at its first independent Australian period and oriented so values above 100 indicate improvement. This shows direction only; it does not claim the indicators are equally important or directly comparable in magnitude.

In [ ]:
TREND_CODES = ['1_1', '1_2', '2_1', '2_2', '2_7', '7_1_DEP', '11_2']
trend = (df.loc[df['country_code'].eq('AUS') & df['year'].le(2024) & df['indicator_code'].isin(TREND_CODES)]
           .drop_duplicates(['indicator_code', 'independent_period'], keep='last')
           .sort_values(['indicator_code', 'year']).copy())
first_values = trend.groupby('indicator_code')['value'].transform('first')
trend['favourable_index'] = trend['value'].div(first_values).mul(100)
lower_is_better = trend['better_direction'].eq('lower')
trend.loc[lower_is_better, 'favourable_index'] = first_values[lower_is_better].div(trend.loc[lower_is_better, 'value']).mul(100)

g = sns.relplot(
    data=trend, x='year', y='favourable_index', col='domain', hue='indicator',
    kind='line', marker='o', facet_kws={'sharex': False, 'sharey': False},
    height=4, aspect=1.15,
)
g.set_axis_labels('Representative year', 'Favourable index (first period = 100)')
g.set_titles('{col_name}')
for ax in g.axes.flat:
    ax.axhline(100, color='grey', linewidth=1, linestyle='--')
plt.show()

## Same-year English-speaking peer sensitivity

Canada, New Zealand, the United Kingdom and the United States are used as one interpretable sensitivity group. They are not assumed to be the only valid peers.

In [ ]:
PEERS = ['CAN', 'NZL', 'GBR', 'USA']
peer_rows = []
for code in TREND_CODES:
    series = df.loc[df['indicator_code'].eq(code) & df['year'].le(2024)]
    aus = (series.loc[series['country_code'].eq('AUS')]
           .drop_duplicates('independent_period', keep='last')
           .sort_values('year'))
    peer = (series.loc[series['country_code'].isin(PEERS)]
            .groupby('year', as_index=False)
            .agg(peer_median=('value', 'median'), peer_count=('country_code', 'nunique')))
    common = aus.merge(peer, on='year').loc[lambda x: x['peer_count'].ge(3)]
    if common.empty:
        continue
    first, last = common.iloc[0], common.iloc[-1]
    peer_rows.append({
        'indicator_code': code, 'indicator': first['indicator'],
        'first_year': int(first['year']), 'last_year': int(last['year']),
        'australia_first': first['value'], 'australia_last': last['value'],
        'peer_median_first': first['peer_median'], 'peer_median_last': last['peer_median'],
        'latest_peer_count': int(last['peer_count']),
    })
peer_summary = pd.DataFrame(peer_rows)
display(peer_summary)

## Recommended research question

> **Has Australia's material progress since 2010 coincided with better social well-being, or has a gap opened relative to comparable countries?**

Suggested subquestions:

1. Did household income and employment improve relative to same-year peers?
2. Were gains accompanied by lower income/wage inequality and fewer long working hours?
3. Did social support and negative affect improve, and was Australia's change unusual among peers?

Before confirmatory analysis, fix the indicator set, peer definitions, primary period and robustness checks. Treat these patterns as exploratory and avoid causal wording.